In [2]:
import pandapower as pp
import simbench as sb
print("Everything working!")

Everything working!


In [4]:
import pandapower as pp
import simbench as sb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load a real German low voltage rural grid
net = sb.get_simbench_net("1-LV-rural1--0-sw")

print("Grid loaded successfully!")
print(f"Number of buses: {len(net.bus)}")
print(f"Number of lines: {len(net.line)}")
print(f"Number of loads (houses): {len(net.load)}")
print(f"Number of transformers: {len(net.trafo)}")

Grid loaded successfully!
Number of buses: 15
Number of lines: 13
Number of loads (houses): 13
Number of transformers: 1


In [5]:
# Run the power flow - simulate electricity flowing through the grid
pp.runpp(net)

# Look at the voltage at every bus
print("=== VOLTAGE AT EVERY BUS ===")
print("(Normal range: 0.95 to 1.05)")
print()
print(net.res_bus[['vm_pu']])

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


=== VOLTAGE AT EVERY BUS ===
(Normal range: 0.95 to 1.05)

       vm_pu
0   1.022034
1   1.024972
2   1.025287
3   1.024739
4   1.019270
5   1.019323
6   1.024562
7   1.025227
8   1.025334
9   1.025936
10  1.026440
11  1.024456
12  1.026532
13  1.022734
42  1.025000


In [6]:
# Check which buses have voltage violations
print("=== BASELINE HEALTH CHECK ===")
print()

v_min = 0.95  # minimum allowed
v_max = 1.05  # maximum allowed

voltages = net.res_bus['vm_pu']

violations = voltages[(voltages < v_min) | (voltages > v_max)]

if len(violations) == 0:
    print("✅ ALL VOLTAGES NORMAL - Grid is healthy!")
    print()
    print(f"Lowest voltage:  {voltages.min():.4f} pu  (limit: {v_min})")
    print(f"Highest voltage: {voltages.max():.4f} pu  (limit: {v_max})")
    print()
    print("This is your BASELINE - the grid before EVs and solar.")
    print("Now we will stress it and watch problems appear.")
else:
    print("❌ VOLTAGE VIOLATIONS FOUND:")
    print(violations)

=== BASELINE HEALTH CHECK ===

✅ ALL VOLTAGES NORMAL - Grid is healthy!

Lowest voltage:  1.0193 pu  (limit: 0.95)
Highest voltage: 1.0265 pu  (limit: 1.05)

This is your BASELINE - the grid before EVs and solar.
Now we will stress it and watch problems appear.


In [7]:
import copy

# Save the original healthy grid
net_baseline = copy.deepcopy(net)

# Add EV chargers to 8 out of 13 houses (60% EV adoption)
# Each EV charger = 11kW (standard home charger in Germany)
ev_buses = [0, 1, 2, 3, 4, 5, 6, 7]  # houses getting EVs

for bus in ev_buses:
    pp.create_load(net, bus=bus, p_mw=0.011, q_mvar=0.0, 
                   name=f"EV_charger_bus{bus}")

# Run power flow with EVs
pp.runpp(net)

# Check results
print("=== GRID WITH EV CHARGERS ===")
print(f"EVs added: {len(ev_buses)} houses")
print()

voltages_ev = net.res_bus['vm_pu']
violations_ev = voltages_ev[(voltages_ev < 0.95) | (voltages_ev > 1.05)]

if len(violations_ev) == 0:
    print("✅ No violations yet")
else:
    print(f"❌ VOLTAGE VIOLATIONS: {len(violations_ev)} buses affected!")
    print(violations_ev)

print()
print(f"Lowest voltage:  {voltages_ev.min():.4f} pu  (limit: 0.95)")
print(f"Highest voltage: {voltages_ev.max():.4f} pu  (limit: 1.05)")

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


=== GRID WITH EV CHARGERS ===
EVs added: 8 houses

✅ No violations yet

Lowest voltage:  1.0038 pu  (limit: 0.95)
Highest voltage: 1.0250 pu  (limit: 1.05)


In [8]:
# Reset grid to baseline
net = copy.deepcopy(net_baseline)

# Add EVs to ALL 13 houses with faster 22kW chargers
all_buses = list(net.load.bus.values)  # all house buses

for bus in all_buses:
    pp.create_load(net, bus=bus, p_mw=0.022, q_mvar=0.0,
                   name=f"EV_22kW_bus{bus}")

pp.runpp(net)

print("=== GRID WITH 22kW EVs ON ALL 13 HOUSES ===")
print()

voltages_ev2 = net.res_bus['vm_pu']
violations_ev2 = voltages_ev2[(voltages_ev2 < 0.95) | (voltages_ev2 > 1.05)]

if len(violations_ev2) == 0:
    print("✅ No violations yet")
else:
    print(f"❌ VOLTAGE VIOLATIONS: {len(violations_ev2)} buses affected!")
    print(violations_ev2)

print()
print(f"Lowest voltage:  {voltages_ev2.min():.4f} pu  (limit: 0.95)")
print(f"Highest voltage: {voltages_ev2.max():.4f} pu  (limit: 1.05)")

# Show line loading - are cables overheating?
print()
print("=== CABLE LOADING ===")
print("(Above 100% = overloaded = overheating risk)")
print()
overloaded = net.res_line[net.res_line['loading_percent'] > 80]
print(f"Lines above 80% capacity: {len(overloaded)}")
print(f"Most loaded cable: {net.res_line['loading_percent'].max():.1f}%")

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


=== GRID WITH 22kW EVs ON ALL 13 HOUSES ===

✅ No violations yet

Lowest voltage:  0.9712 pu  (limit: 0.95)
Highest voltage: 1.0250 pu  (limit: 1.05)

=== CABLE LOADING ===
(Above 100% = overloaded = overheating risk)

Lines above 80% capacity: 0
Most loaded cable: 67.0%


In [9]:
# Reset to baseline
net = copy.deepcopy(net_baseline)

# Add solar panels to ALL 13 houses (5kW each - typical German rooftop)
all_buses = list(net.load.bus.values)

for bus in all_buses:
    pp.create_sgen(net, bus=bus, p_mw=0.005, q_mvar=0.0,
                   name=f"PV_bus{bus}")

pp.runpp(net)

print("=== GRID WITH SOLAR ON ALL 13 HOUSES ===")
print("(Noon on a sunny day - maximum PV production)")
print()

voltages_pv = net.res_bus['vm_pu']
violations_pv = voltages_pv[(voltages_pv < 0.95) | (voltages_pv > 1.05)]

if len(violations_pv) == 0:
    print("✅ No violations yet")
else:
    print(f"❌ VOLTAGE VIOLATIONS: {len(violations_pv)} buses affected!")
    print(violations_pv)

print()
print(f"Lowest voltage:  {voltages_pv.min():.4f} pu  (limit: 0.95)")
print(f"Highest voltage: {voltages_pv.max():.4f} pu  (limit: 1.05)")

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


=== GRID WITH SOLAR ON ALL 13 HOUSES ===
(Noon on a sunny day - maximum PV production)

✅ No violations yet

Lowest voltage:  1.0250 pu  (limit: 0.95)
Highest voltage: 1.0327 pu  (limit: 1.05)


In [10]:
# Reset to baseline
net = copy.deepcopy(net_baseline)

all_buses = list(net.load.bus.values)

# Add 22kW EV chargers to all houses
for bus in all_buses:
    pp.create_load(net, bus=bus, p_mw=0.022, q_mvar=0.0,
                   name=f"EV_22kW_bus{bus}")

# Add 10kW solar to all houses
for bus in all_buses:
    pp.create_sgen(net, bus=bus, p_mw=0.010, q_mvar=0.0,
                   name=f"PV_10kW_bus{bus}")

pp.runpp(net)

print("=== WORST CASE: EVs + SOLAR TOGETHER ===")
print()

voltages_worst = net.res_bus['vm_pu']
violations_worst = voltages_worst[(voltages_worst < 0.95) | (voltages_worst > 1.05)]

if len(violations_worst) == 0:
    print("✅ No violations")
else:
    print(f"❌ VOLTAGE VIOLATIONS: {len(violations_worst)} buses affected!")
    print(violations_worst)

print()
print(f"Lowest voltage:  {voltages_worst.min():.4f} pu  (limit: 0.95)")
print(f"Highest voltage: {voltages_worst.max():.4f} pu  (limit: 1.05)")
print()
print(f"Most loaded cable: {net.res_line['loading_percent'].max():.1f}%")
overloaded = net.res_line[net.res_line['loading_percent'] > 100]
print(f"Overloaded cables: {len(overloaded)}")

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


=== WORST CASE: EVs + SOLAR TOGETHER ===

✅ No violations

Lowest voltage:  0.9941 pu  (limit: 0.95)
Highest voltage: 1.0250 pu  (limit: 1.05)

Most loaded cable: 44.5%
Overloaded cables: 0


In [ ]:
import pandas as pd

print("=== HOSTING CAPACITY ANALYSIS ===")
print("Increasing EV + PV size until violations appear...")
print()

results = []
all_buses = list(net_baseline.load.bus.values)

# Test increasing penetration levels
for kw in range(5, 100, 5):
    net_test = copy.deepcopy(net_baseline)
    
    # Add EVs
    for bus in all_buses:
        pp.create_load(net_test, bus=bus, p_mw=kw/1000, 
                      q_mvar=0.0, name=f"EV_bus{bus}")
    # Add PV
    for bus in all_buses:
        pp.create_sgen(net_test, bus=bus, p_mw=kw/1000, 
                      q_mvar=0.0, name=f"PV_bus{bus}")
    
    pp.runpp(net_test, verbose=False)
    
    v = net_test.res_bus['vm_pu']
    v_min = v.min()
    v_max = v.max()
    line_max = net_test.res_line['loading_percent'].max()
    
    violation = "❌ VIOLATION" if (v_min < 0.95 or v_max > 1.05 or line_max > 100) else "✅ OK"
    
    results.append({
        'EV+PV size (kW)': kw,
        'Min Voltage': round(v_min, 4),
        'Max Voltage': round(v_max, 4),
        'Max Cable %': round(line_max, 1),
        'Status': violation
    })
    
    print(f"{kw:3d} kW per house → V_min={v_min:.4f}  V_max={v_max:.4f}  Cable={line_max:.1f}%  {violation}")

df_results = pd.DataFrame(results)
print()
print("First violation appears at:")
violations_found = df_results[df_results['Status'] == '❌ VIOLATION']
if len(violations_found) > 0:
    print(violations_found.iloc[0])
else:
    print("Grid handles everything tested!")

=== HOSTING CAPACITY ANALYSIS ===
Increasing EV + PV size until violations appear...



numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


  5 kW per house → V_min=1.0193  V_max=1.0265  Cable=41.0%  ✅ OK


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


 10 kW per house → V_min=1.0193  V_max=1.0265  Cable=41.0%  ✅ OK


...

 85 kW per house → V_min=1.0193  V_max=1.0265  Cable=41.0%  ✅ OK


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


 90 kW per house → V_min=1.0193  V_max=1.0265  Cable=41.0%  ✅ OK


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


 95 kW per house → V_min=1.0193  V_max=1.0265  Cable=41.0%  ✅ OK

First violation appears at:
Grid handles everything tested!


In [ ]:
# First let's see what buses actually exist in the grid
net_test = copy.deepcopy(net_baseline)
print("=== GRID BUS LIST ===")
print(net_test.bus)
print()
print("=== EXISTING LOADS (houses) ===")
print(net_test.load[['bus', 'p_mw', 'name']])

=== GRID BUS LIST ===
              name  vn_kv type zone  in_service  \
0    LV1.101 Bus 1    0.4    b  NaN        True   
1    LV1.101 Bus 2    0.4    b  NaN        True   
2    LV1.101 Bus 3    0.4    b  NaN        True   
3    LV1.101 Bus 4    0.4    b  NaN        True   
4    LV1.101 Bus 5    0.4    b  NaN        True   
5    LV1.101 Bus 6    0.4    b  NaN        True   
6    LV1.101 Bus 7    0.4    b  NaN        True   
7    LV1.101 Bus 8    0.4    b  NaN        True   
8    LV1.101 Bus 9    0.4    b  NaN        True   
9   LV1.101 Bus 10    0.4    b  NaN        True   
10  LV1.101 Bus 11    0.4    b  NaN        True   
11  LV1.101 Bus 12    0.4    b  NaN        True   
12  LV1.101 Bus 13    0.4    b  NaN        True   
13  LV1.101 Bus 14    0.4    b  NaN        True   
42   MV1.101 Bus 4   20.0    b  NaN        True   

                                                  geo  min_vm_pu substation  \
0   {"type":"Point", "coordinates":[11.411, 53.6407]}      0.900        NaN   
1  

In [13]:
# Correct bus list - actual house buses
house_buses = list(net_baseline.load.bus.values)
print(f"House buses: {house_buses}")
print(f"Number of houses: {len(house_buses)}")
print()

results = []

# Test EV-only scenarios (no PV cancellation)
for kw in [0, 3, 7, 11, 15, 22, 30, 40, 50]:
    net_test = copy.deepcopy(net_baseline)
    
    # Add EV chargers only
    for bus in house_buses:
        pp.create_load(net_test, bus=bus, 
                      p_mw=kw/1000, 
                      q_mvar=0.0, 
                      name=f"EV_{kw}kW_bus{bus}")
    
    pp.runpp(net_test, verbose=False)
    
    v = net_test.res_bus['vm_pu']
    v_min = round(v.min(), 4)
    v_max = round(v.max(), 4)
    line_max = round(net_test.res_line['loading_percent'].max(), 1)
    
    if v_min < 0.95 or line_max > 100:
        status = "❌ VIOLATION"
    else:
        status = "✅ OK"
    
    results.append({
        'EV_kW': kw,
        'V_min': v_min,
        'V_max': v_max,
        'Cable_%': line_max,
        'Status': status
    })
    
    print(f"EV={kw:2d}kW/house → V_min={v_min}  Cable={line_max}%  {status}")

print()

# Now test PV-only scenarios
print("--- PV ONLY SCENARIOS ---")
for kw in [0, 3, 5, 8, 10, 15, 20, 30]:
    net_test = copy.deepcopy(net_baseline)
    
    # Add solar panels only
    for bus in house_buses:
        pp.create_sgen(net_test, bus=bus,
                      p_mw=kw/1000,
                      q_mvar=0.0,
                      name=f"PV_{kw}kW_bus{bus}")
    
    pp.runpp(net_test, verbose=False)
    
    v = net_test.res_bus['vm_pu']
    v_min = round(v.min(), 4)
    v_max = round(v.max(), 4)
    line_max = round(net_test.res_line['loading_percent'].max(), 1)
    
    if v_max > 1.05 or line_max > 100:
        status = "❌ VIOLATION"
    else:
        status = "✅ OK"
    
    print(f"PV={kw:2d}kW/house → V_max={v_max}  Cable={line_max}%  {status}")

House buses: [np.int64(9), np.int64(7), np.int64(13), np.int64(12), np.int64(8), np.int64(5), np.int64(2), np.int64(0), np.int64(6), np.int64(11), np.int64(10), np.int64(1), np.int64(4)]
Number of houses: 13



numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


EV= 0kW/house → V_min=1.0193  Cable=41.0%  ✅ OK


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


EV= 3kW/house → V_min=1.0132  Cable=34.9%  ✅ OK


...

PV=15kW/house → V_max=1.0476  Cable=71.1%  ✅ OK


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


PV=20kW/house → V_max=1.0563  Cable=80.9%  ❌ VIOLATION


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


PV=30kW/house → V_max=1.0729  Cable=100.4%  ❌ VIOLATION
